# 03 — Exotic Greeks: Asian, Barrier, Lookback

Path-dependent payoffs present additional challenges for Greek estimation:

- **Asian options:** payoff depends on the average of S along the path
- **Barrier options:** payoff is zero if the path crosses the barrier — discontinuous in S₀
- **Lookback options:** payoff depends on the running maximum/minimum

For FD, estimating the delta of a barrier option near the barrier is catastrophic:
bumping S₀ changes which paths survive the barrier — the estimator variance explodes
as S₀ → B.  Malliavin uses the same weight W_T/(σS₀T) and remains stable.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import time

from mgreeks.models.gbm import GeometricBrownianMotion
from mgreeks.payoffs.european import EuropeanCall, DigitalCall
from mgreeks.payoffs.asian import ArithmeticAsianCall
from mgreeks.payoffs.barrier import DownAndOutCall
from mgreeks.payoffs.lookback import FloatingStrikeLookbackCall
from mgreeks.simulation import MonteCarloEngine
from mgreeks.greeks import MalliavinGreeks, FiniteDifferenceGreeks
from mgreeks.greeks.analytical import bs_delta, bs_barrier_delta
from mgreeks.weights.malliavin_weights import delta_weight_path_dependent, delta_weight_gbm

S0, K, T, B = 100.0, 100.0, 1.0, 80.0
r, q, sigma  = 0.05, 0.02, 0.20
n_paths, seed = 50_000, 42
disc = np.exp(-r * T)

model = GeometricBrownianMotion(r=r, q=q, sigma=sigma)
print(f"Parameters: S0={S0}  K={K}  T={T}  B={B}  σ={sigma}")


In [ ]:
## 1. Arithmetic Asian call delta

n_steps = 52   # weekly averaging
engine_asian = MonteCarloEngine(model, n_paths=n_paths, n_steps=n_steps, rng_seed=seed)
mall_asian = MalliavinGreeks(model, engine_asian)
fd_asian   = FiniteDifferenceGreeks(model, engine_asian, bump_size=0.01, bump_type="relative")

asian_payoff = ArithmeticAsianCall(K)

t0 = time.perf_counter()
delta_mall_a = mall_asian.delta(asian_payoff, S0, T)
t_mall = time.perf_counter() - t0

t0 = time.perf_counter()
delta_fd_a   = fd_asian.delta(asian_payoff, S0, T)
t_fd = time.perf_counter() - t0

print("Asian Call Delta (n_steps=52 weekly averaging)")
print(f"  Malliavin : {delta_mall_a['value']:.5f} ± {delta_mall_a['std_error']:.5f}  ({t_mall:.2f}s, 1 sim)")
print(f"  FD (h=1%) : {delta_fd_a['value']:.5f} ± {delta_fd_a['std_error']:.5f}  ({t_fd:.2f}s, 2 sims)")
print(f"  SE ratio FD/Mall: {delta_fd_a['std_error']/delta_mall_a['std_error']:.2f}")


In [ ]:
## 2. Barrier option (DOC) delta near the barrier

n_steps_b = 52
doc_payoff = DownAndOutCall(K, B)

S0_grid = np.linspace(B + 3, 130, 25)
mall_est, mall_se = [], []
fd_est,   fd_se   = [], []

for s0 in S0_grid:
    rng = np.random.default_rng(seed)
    out = model.simulate(s0, T, n_steps_b, n_paths, return_full_paths=True, rng=rng)
    W_T = out["brownian_increments"].sum(axis=1)
    w   = delta_weight_gbm(s0, out["terminal"], sigma, r, q, T, W_T)
    f   = doc_payoff(out["paths"], out["times"])
    samps = disc * f * w
    mall_est.append(float(samps.mean()))
    mall_se.append(float(samps.std(ddof=1) / np.sqrt(n_paths)))

    dh = s0 * 0.01
    rng2 = np.random.default_rng(seed)
    out_up = model.simulate(s0 + dh, T, n_steps_b, n_paths, return_full_paths=True, rng=rng2)
    rng3 = np.random.default_rng(seed)
    out_dn = model.simulate(s0 - dh, T, n_steps_b, n_paths, return_full_paths=True, rng=rng3)
    f_up = doc_payoff(out_up["paths"], out_up["times"])
    f_dn = doc_payoff(out_dn["paths"], out_dn["times"])
    samps_fd = disc * (f_up - f_dn) / (2 * dh)
    fd_est.append(float(samps_fd.mean()))
    fd_se.append(float(samps_fd.std(ddof=1) / np.sqrt(n_paths)))

mall_est, mall_se = np.array(mall_est), np.array(mall_se)
fd_est,   fd_se   = np.array(fd_est),   np.array(fd_se)
analytic = np.array([bs_barrier_delta(s, K, B, T, r, q, sigma) for s in S0_grid])

z = 1.96
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Down-and-Out Call Delta  (B={B}, K={K})", fontsize=13)

ax = axes[0]
ax.axvline(B, color="gray", ls=":", label=f"Barrier B={B}")
ax.plot(S0_grid, analytic, "k-", lw=2, label="Analytical (continuous)")
ax.plot(S0_grid, mall_est, "b-o", ms=4, label="Malliavin")
ax.fill_between(S0_grid, mall_est - z*mall_se, mall_est + z*mall_se, alpha=0.25, color="b")
ax.plot(S0_grid, fd_est, "r--s", ms=4, label="FD (h=1%)")
ax.fill_between(S0_grid, fd_est - z*fd_se, fd_est + z*fd_se, alpha=0.15, color="r")
ax.set_xlabel("S₀"); ax.set_ylabel("Delta"); ax.legend(fontsize=8)
ax.set_title("Estimates ± 95% CI")

ax2 = axes[1]
ax2.axvline(B, color="gray", ls=":")
ax2.semilogy(S0_grid, mall_se, "b-o", ms=4, label="Malliavin SE")
ax2.semilogy(S0_grid, fd_se,   "r--s", ms=4, label="FD SE (h=1%)")
ax2.set_xlabel("S₀"); ax2.set_ylabel("Std Error (log)"); ax2.legend(fontsize=8)
ax2.set_title("SE near barrier: Malliavin flat, FD spikes")

plt.tight_layout()
plt.savefig("03_barrier_sweep.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
## 3. Lookback call delta

engine_lb = MonteCarloEngine(model, n_paths=n_paths, n_steps=52, rng_seed=seed)
mall_lb = MalliavinGreeks(model, engine_lb)
fd_lb   = FiniteDifferenceGreeks(model, engine_lb, bump_size=0.01, bump_type="relative")

lookback = FloatingStrikeLookbackCall()
delta_mall_lb = mall_lb.delta(lookback, S0, T)
delta_fd_lb   = fd_lb.delta(lookback, S0, T)

print("Floating-Strike Lookback Call Delta")
print(f"  Malliavin : {delta_mall_lb['value']:.5f} ± {delta_mall_lb['std_error']:.5f}")
print(f"  FD (h=1%) : {delta_fd_lb['value']:.5f} ± {delta_fd_lb['std_error']:.5f}")


In [ ]:
## 4. Summary table

print(f"{'Payoff':<25} {'Method':<12} {'Delta':>10} {'SE':>10}")
print("=" * 62)
rows = [
    ("European call",        "Malliavin", bs_delta(S0, K, T, r, q, sigma, "call"), 0),
    ("Asian (arith) call",   "Malliavin", delta_mall_a["value"], delta_mall_a["std_error"]),
    ("Asian (arith) call",   "FD (h=1%)", delta_fd_a["value"],   delta_fd_a["std_error"]),
    ("Barrier DOC call",     "Malliavin", mall_est[len(mall_est)//2], mall_se[len(mall_se)//2]),
    ("Barrier DOC call",     "FD (h=1%)", fd_est[len(fd_est)//2],    fd_se[len(fd_se)//2]),
    ("Lookback (float) call","Malliavin", delta_mall_lb["value"], delta_mall_lb["std_error"]),
    ("Lookback (float) call","FD (h=1%)", delta_fd_lb["value"],   delta_fd_lb["std_error"]),
]
for payoff_name, method, val, se in rows:
    print(f"{payoff_name:<25} {method:<12} {val:>10.5f} {se if se else '—':>10}")
print()
print("Note: barrier delta shown at S₀=", round(S0_grid[len(S0_grid)//2], 1))
